In [0]:
# ============================================================
# 06_Final_Data_Validation
# End-to-End Silver → Gold Validation
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 1. Read Silver Clean Table
# ============================================================

silver = spark.table("job_market.silver.jobs_clean")

silver_count = silver.count()

print("=" * 70)
print("FINAL DATA VALIDATION")
print("=" * 70)
print(f"Silver Clean Records : {silver_count:,}")


# ============================================================
# 2. Validate Critical Silver Fields
# ============================================================

critical_nulls = (
    silver
    .select(
        F.sum(F.when(F.col("source").isNull(), 1).otherwise(0)).alias("null_source"),
        F.sum(F.when(F.trim(F.col("company")) == "", 1).otherwise(0)).alias("blank_company"),
        F.sum(F.when(F.col("company").isNull(), 1).otherwise(0)).alias("null_company"),
        F.sum(F.when(F.trim(F.col("job_id")) == "", 1).otherwise(0)).alias("blank_job_id"),
        F.sum(F.when(F.col("job_id").isNull(), 1).otherwise(0)).alias("null_job_id"),
        F.sum(F.when(F.trim(F.col("job_title")) == "", 1).otherwise(0)).alias("blank_job_title"),
        F.sum(F.when(F.col("job_title").isNull(), 1).otherwise(0)).alias("null_job_title"),
        F.sum(F.when(F.col("job_url").isNull(), 1).otherwise(0)).alias("null_job_url"),
        F.sum(F.when(F.col("posted_at").isNull(), 1).otherwise(0)).alias("null_posted_at")
    )
    .collect()[0]
)

critical_null_total = sum(critical_nulls)

print("\n--- Critical Field Validation ---")

if critical_null_total == 0:
    print("PASS: No critical NULL/blank values found.")
else:
    print(f"FAIL: {critical_null_total} critical NULL/blank values found.")
    print(critical_nulls.asDict())


# ============================================================
# 3. Validate Duplicate source + job_id
# ============================================================

duplicate_groups = (
    silver
    .groupBy("source", "job_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n--- Duplicate Validation ---")

if duplicate_groups == 0:
    print("PASS: No duplicate source + job_id groups.")
else:
    print(
        f"FAIL: {duplicate_groups} duplicate source + job_id groups found."
    )


# ============================================================
# 4. Validate Source Distribution
# ============================================================

print("\n--- Source Distribution ---")

display(
    silver
    .groupBy("source")
    .agg(
        F.countDistinct("job_id").alias("job_count"),
        F.countDistinct("company").alias("company_count")
    )
    .orderBy(F.desc("job_count"))
)


# ============================================================
# 5. Validate Gold Job Demand
# ============================================================

job_demand = spark.table("job_market.gold.job_demand")

job_demand_count = (
    job_demand
    .agg(F.sum("job_count").alias("total"))
    .collect()[0]["total"]
)

print("\n--- Job Demand Validation ---")
print(f"Silver Clean Jobs : {silver_count:,}")
print(f"Gold Job Demand   : {job_demand_count:,}")

job_demand_pass = (
    job_demand_count == silver_count
)

if job_demand_pass:
    print("PASS: Job demand reconciles with Silver.")
else:
    print("FAIL: Job demand does not reconcile with Silver.")


# ============================================================
# 6. Validate Location Demand
# ============================================================

location_demand = spark.table("job_market.gold.location_demand")

location_demand_count = (
    location_demand
    .agg(F.sum("job_count").alias("total"))
    .collect()[0]["total"]
)

print("\n--- Location Demand Validation ---")
print(f"Silver Clean Jobs : {silver_count:,}")
print(f"Location Demand   : {location_demand_count:,}")

location_pass = (
    location_demand_count == silver_count
)

if location_pass:
    print("PASS: Location demand reconciles with Silver.")
else:
    print("FAIL: Location demand does not reconcile with Silver.")


# ============================================================
# 7. Validate Workplace Demand
# ============================================================

workplace_demand = spark.table(
    "job_market.gold.workplace_demand"
)

workplace_count = (
    workplace_demand
    .agg(F.sum("job_count").alias("total"))
    .collect()[0]["total"]
)

print("\n--- Workplace Demand Validation ---")
print(f"Silver Clean Jobs : {silver_count:,}")
print(f"Workplace Demand  : {workplace_count:,}")

workplace_pass = (
    workplace_count == silver_count
)

if workplace_pass:
    print("PASS: Workplace demand reconciles with Silver.")
else:
    print("FAIL: Workplace demand does not reconcile with Silver.")


# ============================================================
# 8. Validate Hiring Trends
# ============================================================

hiring_trends = spark.table(
    "job_market.gold.hiring_trends"
)

posted_jobs_count = (
    silver
    .filter(F.col("posted_at").isNotNull())
    .count()
)

hiring_trends_count = (
    hiring_trends
    .agg(F.sum("job_count").alias("total"))
    .collect()[0]["total"]
)

print("\n--- Hiring Trends Validation ---")
print(f"Silver Jobs with posted_at : {posted_jobs_count:,}")
print(f"Hiring Trends Job Count    : {hiring_trends_count:,}")

hiring_trends_pass = (
    hiring_trends_count == posted_jobs_count
)

if hiring_trends_pass:
    print("PASS: Hiring trends reconcile with posted jobs.")
else:
    print("FAIL: Hiring trends do not reconcile.")


# ============================================================
# 9. Validate Skill Demand
# ============================================================

skill_demand = spark.table(
    "job_market.gold.skill_demand"
)

invalid_skill_counts = (
    skill_demand
    .filter(
        (F.col("job_count") <= 0) |
        (F.col("job_count") > silver_count)
    )
    .count()
)

print("\n--- Skill Demand Validation ---")

if invalid_skill_counts == 0:
    print(
        "PASS: Every skill job_count is between 1 and total Silver jobs."
    )
else:
    print(
        f"FAIL: {invalid_skill_counts} invalid skill counts found."
    )


# ============================================================
# 10. Validate Company Skill Demand
# ============================================================

company_skill_demand = spark.table(
    "job_market.gold.company_skill_demand"
)

invalid_company_skill_counts = (
    company_skill_demand
    .filter(
        (F.col("job_count") <= 0) |
        (F.col("job_count") > silver_count)
    )
    .count()
)

print("\n--- Company Skill Demand Validation ---")

if invalid_company_skill_counts == 0:
    print(
        "PASS: Company-skill job counts are within valid range."
    )
else:
    print(
        f"FAIL: {invalid_company_skill_counts} invalid company-skill counts found."
    )


# ============================================================
# 11. Validate Gold Tables Exist and Contain Data
# ============================================================

gold_tables = [
    "job_demand",
    "skill_demand",
    "company_skill_demand",
    "location_demand",
    "location_job_demand",
    "workplace_demand",
    "hiring_trends"
]

print("\n--- Gold Table Availability ---")

gold_table_results = []

for table_name in gold_tables:

    full_name = f"job_market.gold.{table_name}"

    try:
        count = spark.table(full_name).count()

        status = "PASS" if count > 0 else "FAIL"

        gold_table_results.append(
            (table_name, count, status)
        )

        print(
            f"{table_name:<25} | "
            f"{count:>8,} rows | {status}"
        )

    except Exception as e:

        gold_table_results.append(
            (table_name, 0, "FAIL")
        )

        print(
            f"{table_name:<25} | "
            f"TABLE NOT FOUND | FAIL"
        )


# ============================================================
# 12. Overall Validation Result
# ============================================================

gold_tables_pass = all(
    status == "PASS"
    for _, _, status in gold_table_results
)

all_validations_pass = (
    critical_null_total == 0
    and duplicate_groups == 0
    and job_demand_pass
    and location_pass
    and workplace_pass
    and hiring_trends_pass
    and invalid_skill_counts == 0
    and invalid_company_skill_counts == 0
    and gold_tables_pass
)


# ============================================================
# 13. Final Summary
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Critical Fields       : "
    f"{'PASS' if critical_null_total == 0 else 'FAIL'}"
)

print(
    f"Duplicate Check       : "
    f"{'PASS' if duplicate_groups == 0 else 'FAIL'}"
)

print(
    f"Job Demand            : "
    f"{'PASS' if job_demand_pass else 'FAIL'}"
)

print(
    f"Location Demand       : "
    f"{'PASS' if location_pass else 'FAIL'}"
)

print(
    f"Workplace Demand      : "
    f"{'PASS' if workplace_pass else 'FAIL'}"
)

print(
    f"Hiring Trends         : "
    f"{'PASS' if hiring_trends_pass else 'FAIL'}"
)

print(
    f"Skill Demand          : "
    f"{'PASS' if invalid_skill_counts == 0 else 'FAIL'}"
)

print(
    f"Company Skill Demand  : "
    f"{'PASS' if invalid_company_skill_counts == 0 else 'FAIL'}"
)

print(
    f"Gold Tables            : "
    f"{'PASS' if gold_tables_pass else 'FAIL'}"
)

print("=" * 70)

if all_validations_pass:
    print("✓ FINAL VALIDATION PASSED")
    print("✓ Silver → Gold pipeline is consistent.")
else:
    print("✗ FINAL VALIDATION FAILED")
    print("✗ Review the failed validation(s) above.")

print("=" * 70)

FINAL DATA VALIDATION
Silver Clean Records : 2,221

--- Critical Field Validation ---
PASS: No critical NULL/blank values found.

--- Duplicate Validation ---
PASS: No duplicate source + job_id groups.

--- Source Distribution ---


source,job_count,company_count
greenhouse,1583,5
lever,638,4



--- Job Demand Validation ---
Silver Clean Jobs : 2,221
Gold Job Demand   : 2,221
PASS: Job demand reconciles with Silver.

--- Location Demand Validation ---
Silver Clean Jobs : 2,221
Location Demand   : 2,221
PASS: Location demand reconciles with Silver.

--- Workplace Demand Validation ---
Silver Clean Jobs : 2,221
Workplace Demand  : 2,221
PASS: Workplace demand reconciles with Silver.

--- Hiring Trends Validation ---
Silver Jobs with posted_at : 2,221
Hiring Trends Job Count    : 2,221
PASS: Hiring trends reconcile with posted jobs.

--- Skill Demand Validation ---
PASS: Every skill job_count is between 1 and total Silver jobs.

--- Company Skill Demand Validation ---
PASS: Company-skill job counts are within valid range.

--- Gold Table Availability ---
job_demand                |    1,911 rows | PASS
skill_demand              |       44 rows | PASS
company_skill_demand      |      210 rows | PASS
location_demand           |      351 rows | PASS
location_job_demand       |    2